In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options


import pandas as pd

In [129]:
login_url = "https://www.tilastopaja.info/login.php"

options = Options()

options.page_load_strategy = 'normal'
options.add_argument('--headless')        # Run in headless mode
options.add_argument('--window-size=1920,1080')  # Optional: set window size

driver = webdriver.Chrome(options=options)
driver.get(login_url)

python(14603) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
The chromedriver version (134.0.6998.165) detected in PATH at /opt/homebrew/bin/chromedriver might not be compatible with the detected chrome version (135.0.7049.85); currently, chromedriver 135.0.7049.95 is recommended for chrome 135.*, so it is advised to delete the driver in PATH and retry
python(14648) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [130]:
# Find input fields and login
driver.find_element(By.NAME, "user").send_keys(os.environ["TILASTOPAJA_USER"])
driver.find_element(By.NAME, "password").send_keys(os.environ["TILASTOPAJA_PASS"] + Keys.RETURN)
driver.find_element(By.XPATH,"//input[@type='button' and @value='Login']").click()

In [131]:
roster_file = "/Users/mariograndi/Desktop/Sports Data/Data/Roster/all_athletes_roster.h5"
events_file = "/Users/mariograndi/Desktop/Sports Data/Data/Events.h5"
all_athletes = pd.read_hdf(roster_file)
all_events = pd.read_hdf(events_file)

In [132]:
if True:
    cols = [element for element in all_athletes.columns if "outdoor" in element or "indoor" in element]
    athletes_subset = []
    for col in cols: 
        athletes_subset.append( all_athletes.loc[all_athletes[col]==1].iloc[0]['PID'])

    athletes_subset = list(set(athletes_subset))

In [144]:
athlete_results = {}

athlete_results["index"] = []
athlete_results["data"] = []

event_list = {}
i=0
for name,pid in all_athletes.loc[all_athletes["PID"].isin(athletes_subset)][["Athlete Name",'PID']].values:
# for name,pid in all_athletes[["Athlete Name",'PID']].values:
    i+=1
    # print("{}/{} - {} ({}){}".format(i,len(all_athletes),name,pid," "*10),end="\r")
    print("{}/{} - {} ({}){}".format(i,len(all_athletes.loc[all_athletes["PID"].isin(athletes_subset)]),name,pid," "*10),end="\r")
    athlete_url = all_athletes.loc[all_athletes['PID']==pid,'Athlete URL'].values[0]
    driver.get(athlete_url)
    soup = BeautifulSoup(driver.page_source,"html.parser")

    athlete_results["index"].append(pid)
    
    for location in ['Outdoor','Indoor']:
        tables = soup.find_all("div",id=lambda value: value and value.startswith(location+"x"))
        for table in tables:
            event = table['id'].split("x")[-1]
            event_name = table.find_all_previous('tr',limit=3)[-1].text
            rows = table.find_all("tr")

            year=None
            io=None
            skip=False
            
            for row in rows: 
                columns = row.find_all("td")
                datum = [col.text.strip() for col in columns]
                if len(datum) == 1:
                    if datum[0].isnumeric():
                        year = int(datum[0])
                        skip=True
                    elif datum[0] in ['Outdoor','Indoor']:
                        io = datum[0]
                        skip=True

                    else:
                        skip=True
                else:
                    skip=False

                if not skip:
                    a = datum[0].split()
                    a+=datum[1:]
                    # print(len(datum))
                    # print(event,event_name,year,io,*a)

                    if (event,event_name,io) not in event_list:
                        event_list[(event,event_name,io)] = datum
                    
                    if len(datum) > len(event_list[(event,event_name,io)]):
                        event_list[(event,event_name,io)] = datum

                    #athlete_results["data"].append( [name,event,event_name,io,year,*a] )

In [203]:
import numpy as np
for i in range(len(np.array(list(event_list.values())))):
    if len(np.array(list(event_list.values()))[i][0].split("\xa0"))>=4:
        print(np.array(list(event_list.values()))[i][0].split("\xa0"))

['14.35', '', '+1.1', 'WB']
['30.97', '', '', 'AR']
['3:04.06', '', '', 'AU20R']
['10.05 OT', '', '', 'NR']
['10.05 OT', '', '', 'NR']
['1:22.11', '', '', 'WR']
['47.72', '', '', 'NU20R']
['48.06', '', '', 'NR']
['10.35 OT', '', '', 'NR']
['33.43', '', '', '', 'NJR']
['1:17:25.6h', '', '', 'WR']
['20:15.71', '', '', 'NR']


In [123]:
topp_100m = pd.read_hdf("/Users/mariograndi/Desktop/Sports Data/Data/Events/top_performance_100m_outdoor_male.h5")
topp_100m

,Rank,Score,Wind,Record,Athlete Name,Athlete Country,Athlete Date of Birth,Position,Event,Event Location,Date of Event,Indoor,Athlete Sex,PID,Athlete URL
0,1,9.58,+0.9,WR,Usain Bolt,JAM,21 Aug 86,1,WC,Berlin,16 Aug 2009,0,1,45032,https://www.tilastopaja.info/db/at.php?Sex=1&I...
1,2,9.69,+2.0,AR,Tyson Gay,USA,9 Aug 82,1,GoldenGP,Shanghai,20 Sep 2009,0,1,45173,https://www.tilastopaja.info/db/at.php?Sex=1&I...
2,2,9.69,-0.1,,Yohan Blake,JAM,26 Dec 89,1,Athletissima,Lausanne,23 Aug 2012,0,1,69837,https://www.tilastopaja.info/db/at.php?Sex=1&I...
3,4,9.72,+0.2,,Asafa Powell,JAM,23 Nov 82,1,Athletissima,Lausanne,2 Sep 2008,0,1,4109,https://www.tilastopaja.info/db/at.php?Sex=1&I...
4,5,9.74,+0.9,,Justin Gatlin,USA,10 Feb 82,1,Diamond,Doha,15 May 2015,0,1,23308,https://www.tilastopaja.info/db/at.php?Sex=1&I...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7943,7519,10.48A,+0.3,,Elton Hoeseb,NAM,27 Apr 03,2,,Windhoek,8 Feb 2025,0,1,826008,https://www.tilastopaja.info/db/at.php?Sex=1&I...
7944,7519,10.48,+0.8,,Kimani Fletcher,JAM,4 Mar 01,1r7,,Spanish Town,8 Feb 2025,0,1,10048963,https://www.tilastopaja.info/db/at.php?Sex=1&I...
7945,7519,10.48,+1.6,,Justin Steele,USA,,1rB,,Irvine CA,14 Mar 2025,0,1,339334,https://www.tilastopaja.info/db/at.php?Sex=1&I...
7946,7519,10.48,+1.0,,Ryan Mann,USA,,2,,San Luis Obispo CA,22 Mar 2025,0,1,818896,https://www.tilastopaja.info/db/at.php?Sex=1&I...


In [124]:
event_list[('40', 'Outdoor')]

['10.08',
 '-1.4',
 'DQ',
 'IAAF',
 'Rule',
 '32.2.a',
 'h1',
 'OG',
 'London',
 '4 Aug']

In [11]:
# # soup.findAll('div',class_="center")[0]
# tab = soup.find('div',id="progressionDiv").findAll('table')[0]

# rows = tab.findAll('tr')
# event_found=False
# year_found= False
# io_found = False
# for i,row in enumerate(rows):
#     columns = row.find_all("td")
#     datum = [col.text.strip() for col in columns]
#     print(i,datum)

#     if datum[0] in all_events['Event Name']:
#         event_found=True
#         year_found=False
#         io_found=False

#         event_name=datum[0]
    
#     if event_found and not io_found and datum[0] in ["Outdoor","Indoor"]:
#         io_found=True
#         io = 1 if datum[0] == "Indoor" else 0 

#     if event_found and not year_found and datum[0].isdigit():
#         year_found = True
#         year = datum[0]

#     if event_found and year_found and io_found:
        




        


In [ ]:
tables = soup.find_all("div",id=lambda value: value and value.startswith("Indoorx"))

rows = tables.find_all("tr")

for row in rows:
    columns = row.find_all("td")
    datum = [col.text.strip() for col in columns]
    print(datum)

IndexError: list index out of range

In [99]:
driver.get("https://www.tilastopaja.info/db/at.php?Sex=1&ID=13845")
soup = BeautifulSoup(driver.page_source,"html.parser")

In [6]:
test = soup.find_all("div",id=lambda value: value and value.startswith("Outdoorx"))
for table in test:
    event_io = table['id'].split("x")[0]
    event_io_int = 1 if table['id'].split("x")[0] == "Indoor" else 0 

    event_id = int(table['id'].split("x")[-1])
    if event_id in all_events['Event ID'].values:
        print (all_events.loc[(all_events['Event ID']==event_id)&(all_events['Indoor']==str(event_io_int))]["Event Name"].values,event_io,event_id)
    else:
        print(event_id)


39
['100m'] Outdoor 40
48
49
['200m'] Outdoor 50
['300m'] Outdoor 60
['400m'] Outdoor 70
['4 x 100m'] Outdoor 560
['4 x 400m'] Outdoor 580


In [ ]:
athlete_results = {}

for location in ['Outdoor','Indoor']:
    tables = soup.find_all("div",id=lambda value: value and value.startswith(location+"x"))
    for table in tables:
        event = table['id'].split("x")[-1]
        event_name = table.find_all_previous('tr',limit=3)[-1].text
        rows = table.find_all("tr")

        year=None
        io=None
        skip=False
        for row in rows: 
            columns = row.find_all("td")
            datum = [col.text.strip() for col in columns]
            if len(datum) == 1:
                if datum[0].isnumeric():
                    year = int(datum[0])
                    skip=True
                elif datum[0] in ['Outdoor','Indoor']:
                    io = datum[0]
                    skip=True

                else:
                    skip=True
            else:
                skip=False

            if not skip:
                a = datum[0].split()
                a+=datum[1:]
                print(event,event_name,year,io,*a)
                athlete_results[]






39 100y 2011 Outdoor 9.14+ 1 GSpike Ostrava 31 May
39 100y 2012 Outdoor 9.29+ -0.8 1 GSpike Ostrava 25 May
40 100m 2007 Outdoor 10.03 +0.7 1r1 Vard Réthimno 18 Jul
40 100m 2008 Outdoor 10.03 +1.8 1r1 Classics Spanish Town 8 Mar
40 100m 2008 Outdoor 9.76 +1.8 1r2 Jamaica Inv Kingston 3 May
40 100m 2008 Outdoor 9.92 +0.6 1r2 HamptonG Port of Spain 17 May
40 100m 2008 Outdoor 9.72 +1.7 WR 1 Reebok New York NY 31 May
40 100m 2008 Outdoor 10.19 +1.0 1h1 NC Kingston 27 Jun
40 100m 2008 Outdoor 10.40 -2.0 1s2 NC Kingston 28 Jun
40 100m 2008 Outdoor 9.85 -0.1 1 NC Kingston 28 Jun
40 100m 2008 Outdoor 9.89 +0.4 2 DNG Stockholm/S 22 Jul
40 100m 2008 Outdoor 10.20 -0.2 1h1 OG Beijing 15 Aug
40 100m 2008 Outdoor 9.92 +0.1 1q4 OG Beijing 15 Aug
40 100m 2008 Outdoor 9.85 -0.1 1s1 OG Beijing 16 Aug
40 100m 2008 Outdoor 9.69 0.0 WR 1 OG Beijing 16 Aug
40 100m 2008 Outdoor 9.83 -0.5 1 WK Zürich 29 Aug
40 100m 2008 Outdoor 9.77 -1.3 1 VD Bruxelles 5 Sep
40 100m 2009 Outdoor 9.93 +2.3 1r1 Classics Spanis

In [ ]:
row.find_all('td')

[<td>21.33  +2.2   </td>,
 <td>1-19</td>,
 <td></td>,
 <td>Spanish Town</td>,
 <td>1 Mar</td>]

In [155]:
if 10 in all_events['Event ID'].values:
    print(True)

In [12]:
soup

<html><head><script async="" src="https://www.googletagmanager.com/gtag/js?id=G-95HH6N92RL"></script>
<script>
  window.dataLayer = window.dataLayer || [];
  function gtag(){dataLayer.push(arguments);}
  gtag('js', new Date());

  gtag('config', 'G-95HH6N92RL');
</script>
<meta content="width=device-width" name="viewport"/>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="Track and field statistics, athletics statistics, results." name="Description"/>
<title>Tilastopaja Oy Track and field statistics</title>
<link href="../main.css" rel="stylesheet" type="text/css"/>
<link href="defaultat.css?v=3" media="screen" rel="stylesheet" type="text/css"/>
<script src="https://ajax.googleapis.com/ajax/libs/jquery/3.3.1/jquery.min.js"></script>
<script>
function changeme(id) {
       if (document.getElementById(id).style.display == "none") {
            document.getElementById(id).style.display = "block";
       } else {
            document.getElementById(id).st